# RunLoadTest — Orchestrator Notebook

This is the **driver (parent) notebook**. It launches one or more copies of
`RunPerfScenario` in parallel using Fabric's `notebookutils.notebook.runMultiple()`.

## How it works
1. **Cell 1 (this cell)** — documentation.
2. **Cell 2** — Spark session configuration (vCores). Increase this when you need
   more concurrent threads. Recommended values: 4, 8, 16, 32, 64.
3. **Cell 3** — All the logic:
   - Reads configuration (workspace, dataset, XMLA endpoint, concurrency, etc.).
   - Auto-discovers Performance Analyzer JSON files from the lakehouse.
   - Builds a DAG (Directed Acyclic Graph) of parallel notebook activities.
   - Launches them via `runMultiple` and prints the outcome.

## Scaling up
To run with higher concurrency you'll need to scale up the Spark session.
By default it runs with **4 vCores**. Edit Cell 2 to request more:

```
%%configure -f
{
    "vCores": 16
}
```

This will increase notebook startup time as a larger VM must be provisioned.


In [ ]:
# Spark session configuration — increase vCores for higher concurrency.
# Recommended values: 4, 8, 16, 32, 64.  Fabric auto-allocates memory
# proportionally to the chosen vCores value.
%%configure -f
{
    "vCores": 4
}


In [ ]:
import sempy.fabric
import os
import time


# ─────────────────────────────────────────────────────────────
#  CONFIG — edit this section before every test run
# ─────────────────────────────────────────────────────────────
# load_test_name : a human-readable label written into every log row
# dataset        : the Power BI semantic model (dataset) you want to stress-test
# xmla_endpoint  : the XMLA read/write endpoint of the workspace that hosts the
#                  dataset.  Set to None to use the current workspace.
# workspace      : display name of the workspace that contains the dataset
# home_workspace : workspace where *this* orchestrator notebook lives — Fabric
#                  needs it to locate the child RunPerfScenario notebook
load_test_name = "Query DirectQuery CRWEG-Dragos DB RLS"
dataset        = "Query DirectQuery CRWEG-Dragos"
xmla_endpoint  = "powerbi://api.powerbi.com/v1.0/myorg/PBI%20MRL%20DEV%20CRWEG%20CDL%20SS"
workspace      = "PBI MRL DEV CRWEG CDL SS"
home_workspace = "Stress Testing - Internal"

# concurrent_threads : how many parallel RunPerfScenario notebooks to launch
# delay_sec          : think-time (seconds) each virtual user waits between queries
# iterations         : how many times each virtual user replays the full query set
concurrent_threads = 1
delay_sec          = 4
iterations         = 1

# Fall back to the current workspace name if 'workspace' is empty/None
workspace = workspace or notebookutils.runtime.context["currentWorkspaceName"]

# ─────────────────────────────────────────────────────────────
#  DISCOVER QUERY FILES
# ─────────────────────────────────────────────────────────────
# Performance Analyzer JSON exports must be placed in
#   /lakehouse/default/Files/PerfScenarios/Queries/<workspace>/
# Each JSON contains the DAX queries recorded from a Power BI report.
# If you have multiple JSON files, they are distributed round-robin
# across the virtual users (threads).
query_folder = f"/lakehouse/default/Files/PerfScenarios/Queries/{workspace}"
query_files  = sorted(
    f"{query_folder}/{f}"
    for f in os.listdir(query_folder)
    if f.endswith(".json")
)


# ─────────────────────────────────────────────────────────────
#  BUILD & RUN THE LOAD TEST
# ─────────────────────────────────────────────────────────────

# Timestamp used as part of the folder name so each run gets its own log folder.
# Use a fixed string while iterating, or uncomment the line below for auto-generated.
ts         = "20260304_1"
# ts       = time.strftime("%Y%m%d-%H%M%S")   # uncomment for unique folder per run
loadtestId = f"{load_test_name}-{ts}"
print(loadtestId)

# Create the output folder in the lakehouse where all CSV logs will be written
folder_path = f"/lakehouse/default/Files/PerfScenarios/logs/{workspace}/{loadtestId}"
notebookutils.fs.mkdirs(folder_path)

# Optional: list specific user UPNs if you need to test Row-Level Security (RLS)
# with impersonation. Each thread will cycle through users round-robin.
# Leave empty to run as the current notebook user.
users = []
# users = [
#     "tartau@merck.com",
#     "psaar@merck.com",
#     "dkapadia@merck.com",
#     "jasipovi@merck.com",
#     "padhia@merck.com",
# ]

# Base arguments passed to every child RunPerfScenario notebook.
# Per-thread overrides (threadId, perf_analyzer_filename, etc.) are applied below.
args = {
    "xmla_endpoint"          : xmla_endpoint,
    "workspace"              : workspace,
    "perf_analyzer_filename" : None,            # set per-thread below
    "model"                  : dataset,
    "roles"                  : None,
    "customdata"             : None,
    "effective_username"     : None,            # set per-thread below if users list is populated
    "iterations"             : iterations,
    "delay_sec"              : delay_sec,
    "loadtestId"             : loadtestId,
    "threadId"               : 0,              # set per-thread below
    "concurrent_threads"     : concurrent_threads,
    "useRootDefaultLakehouse": True,
}

# Template for one notebook activity inside the Fabric DAG.
# "path" must match the notebook name in the workspace.
activity = {
    "name"                   : "RunPerfScenario",
    "path"                   : "RunPerfScenario",
    "timeoutPerCellInSeconds": 9000,
    "args"                   : {},
    "workspace"              : home_workspace,
    "retry"                  : 0,
    "retryIntervalInSeconds" : 0,
    "dependencies"           : [],
}

# The DAG defines which notebook activities run and how many can run in parallel.
DAG = {
    "activities"      : [],
    "timeoutInSeconds": 43200,          # 12-hour global timeout
    "concurrency"     : concurrent_threads,
}

# Create one activity per virtual user (thread).
# Each gets its own threadId, query file, and optionally an impersonated user.
for i in range(concurrent_threads):
    a = activity.copy()
    a["name"] = f"{a['name']}_{i}"

    this_args = args.copy()
    this_args["threadId"]               = i
    this_args["perf_analyzer_filename"] = query_files[i % len(query_files)]
    this_args["folder_path"]            = folder_path

    if users:
        user = users[i % len(users)]
        this_args["effective_username"] = user
        this_args["username"]           = user.split("@")[0]

    a["args"] = this_args
    DAG["activities"].append(a)

# ─────────────────────────────────────────────────────────────
#  EXECUTE
# ─────────────────────────────────────────────────────────────
# runMultiple launches all activities in parallel (up to the concurrency limit).
# If any child notebook fails, the exception object still contains partial results.
print("running load test")

try:
    results = notebookutils.notebook.runMultiple(DAG)
except Exception as e:
    results = e.result
    for notebook_name, result in results.items():
        if result.get("exception"):
            print(f"Notebook {notebook_name} failed:")
            print(result["exception"])

print("load test complete")
